# Semenov et al. (2026) on DANDI:001526

Paper: Semenov, Silkuniene, Silkunas, Pakhomova, Bixler, Ibey, Kochunov and Pakhomov (2026),
"Sub-microsecond optical measurements of cell membrane charging and lesioning by pulsed electric
fields", *Bioelectrochemistry* 171, 109283, doi:10.1016/j.bioelechem.2026.109283.

Dataset: DANDI:001526, "Assessment of Post-Electroporation Suitability of CHO Cells for
Polarization Studies Using Strobe Fluorescence Microscopy", version **0.261006.1851**
(doi:10.48324/dandi.001526/0.261006.1851). The paper cites no version (its Data availability
statement says "Data will be made available on request"), so the most recent published version is
pinned. It is the only published version; the draft has the same 243 assets and size, so there is
no earlier version that could match the claims file better.

Claims: `claims/001526/10.1016-j.bioelechem.2026.109283.yaml`, written from the version of record
(main text, Methods, Results, Discussion and legends; the paper has no supplement and links no code).

The dataset holds Di-8-ANEPPS strobe images of 19 cells in two sessions: 11 cells on 2025-05-01
with 1 us pulses alternating between 50 and 70 V, then 250 V, then 50 and 70 V again (the
protocol of Fig. 2C and D, 150, 230 and 1000 V/cm), and 8 cells on 2025-05-06 with 0.3 us pulses
escalating from 100 to 800 V (the protocol of Fig. 4D, 357 to 3320 V/cm). No FluoVolt, Fluo-4,
Gd3+, two-pulse or 1 us escalation recordings are in it.

| Claim | Tested here |
|---|---|
| fig2cd-nonelectroporating-stable | tested (2025-05-01) |
| fig2c-1000-within-set | tested (2025-05-01) |
| fig2d-lasting-reduction | tested (2025-05-01) |
| fig4d-short-pulses-same-limit | tested (2025-05-06) |
| fig5b-eq2-sensitivity | tested (a computation from Eq. 2, no recordings) |
| fig1-pattern-poles-leaky, fig4b-first10-cathodic, fig4b-asymmetric-decline, fig4b-di8-threshold-570, fig4c-equator-cosine, fig7b-conditioning-sublinear | blocked, data missing: the Di-8-ANEPPS 1 us escalation (Fig. 4B group) is not in the dataset |
| fig2a-fluovolt-saturation, fig2a-fluovolt-baseline-stable, fig2a-fluovolt-stable-electroporating, fig4a-fluovolt-limits | blocked, data missing: no FluoVolt recordings |
| fig2b-di8-climbs-to-570, fig2b-di8-bleaching-43, fig2b-di8-decline-within-set | blocked, data missing: the Fig. 2B escalation is not in the dataset |
| fig3-dye-slopes, fig3-fluovolt-nonlinear | blocked, data missing: no FluoVolt and no 43 to 360 V/cm calibration set; also no pixel size and no tau fit |
| fig5a-gd-anodic-extension, fig5a-gd-linear-to-780, fig5a-gd-cathodic, fig5a-gd-significant-points, fig5c-gd-reduces-conductivity | blocked, data missing: no Gd3+ experiments |
| fig6a-single-pulse-ca, fig6b-train-ca | blocked, data missing: no Fluo-4 imaging |
| fig7a-test-pulse-cosine, fig7b-test-pulse-recovery, fig7c-fast-resealing | blocked, data missing: no two-pulse experiments |

No random numbers are used anywhere in this notebook, so no seed is needed.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import strobe

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("dandiset", strobe.DANDISET, "version", strobe.VERSION)
t0 = time.time()
files = strobe.table()
print(f"metadata of {len(files)} files read in {time.time() - t0:.0f} s")

## Loading

Every file is streamed once and reduced to the 6-pixel-wide perimeter trace of the analysed cell
(360 points, 100 images), the per-image median of a cell-free annulus around it, and the trace
of the cell's background recording where there is one (see `strobe.py` for every parameter and
where it comes from). This takes about ten minutes.

In [ ]:
t0 = time.time()
res = strobe.extract(files)
print(f"{len(res)} files reduced in {(time.time() - t0) / 60:.1f} min")

## Data check

In [ ]:
cells = files[files.kind != "background"].copy()
pef = files[files.kind == "pef"].copy()
print("files by session and kind:")
print(files.groupby(["session_id", "kind"]).size().to_string())
print("\ncells per session:", files[files.kind != "background"].groupby("session_id").cell.nunique().to_dict())
print("\nPEF recordings by pulse, amplitude and field label:")
print(pef.groupby(["duration_us", "volts", "field"], dropna=False).size().to_string())
print("\nrecordings per cell (sham + PEF), and background recordings:")
print(pd.crosstab(files.cell, files.kind).to_string())
print("\nimage stacks:", files["shape"].astype(str).value_counts().to_dict(), "dtype", files.dtype.unique(),
      "unit", files.unit.unique())
print("imaging rate (Hz):", files.rate.unique(), "; electrode distance in the files (mm):", files.electrode_mm.unique(),
      "(paper, section 2.2: 1.6 mm spacing)")
st = files.starting_time
print(f"starting_time ranges {st.min():.0f} to {st.max():.0f} s: these are Unix times, not seconds from session start")
print("\n1 us sequence of amplitudes per cell, in time order (V):")
for cell, g in pef[pef.duration_us == 1].groupby("cell"):
    print(" ", cell, list(g.volts.astype(int)))
print("\n0.3 us sequence of amplitudes per cell (V):")
for cell, g in pef[pef.duration_us == 0.3].groupby("cell"):
    print(" ", cell, list(g.volts.astype(int)))
gen = files.description.str.split("#####").str[-1].str.strip().unique()
print("\ndistinct general descriptions in the files:", len(gen))
print(" ", gen[0][:300], "...")

geo = pd.DataFrame([dict(cell=r.cell, session=r.session_id, radius_px=res[r.asset_id]["radius"],
                         shift_r=res[r.asset_id]["shift"][0], shift_c=res[r.asset_id]["shift"][1],
                         n_annulus=res[r.asset_id]["n_annulus"]) for r in cells.itertuples()])
print("\ncell geometry (traced on the sham recording; later recordings moved by the measured drift):")
print(geo.groupby("cell").agg(radius_px=("radius_px", "first"), max_abs_shift_px=("shift_r", lambda v: 0),
                              annulus_px_min=("n_annulus", "min")).assign(
    max_abs_shift_px=geo.assign(s=np.maximum(geo.shift_r.abs(), geo.shift_c.abs())).groupby("cell").s.max()).to_string())

fm = np.array([res[a]["frame_mean"] for a in cells.asset_id])
ratio = fm[:, 0] / fm[:, 1:40].mean(1)
print(f"\nfirst image / mean of images 2-40 (whole frame): median {np.median(ratio):.3f}, range {ratio.min():.3f} to "
      f"{ratio.max():.3f}; files with ratio < 0.95: {(ratio < 0.95).sum()} of {len(ratio)}")

Checks of the metadata against the data:

- The files give pulse amplitudes in volts, not the field strengths in V/cm that the paper and the
  claims use. The amplitudes are matched to the paper's labels by order, which is monotonic in both:
  50, 70 and 250 V are the 150, 230 and 1000 V/cm of Fig. 2C, whose sequence (four 150/230
  alternations, 1000, then two more alternations) the 2025-05-01 files follow; 100 to 800 V at 0.3
  us are the nine fields of Fig. 4D (357 to 3320 V/cm, read from the figure's axis labels). One cell
  (2025-05-06 c1) has a 450 V set with no counterpart in Fig. 4D; it is left out of the per-field
  means.
- The files state an electrode distance of 1.00 mm; the paper (section 2.2) states 1.6 mm.
- Every file of both sessions carries the same general description ("Cells were exposed to a 50, 70
  or 250V single pulse ... 13 recordings"), which describes the 2025-05-01 protocol only; the
  2025-05-06 files hold 0.3 us pulses of 100 to 800 V. Each file's own first sentence (pulse and
  amplitude) agrees with the sequence of recordings and is the one used. The description also says
  "a single ... pulse", while each set holds 60 pulsed images (below).
- `starting_time` holds Unix times, not times relative to the session start.
- No pixel size is given, so the cell radius is known only in pixels.
- The files do not say which cell in the field of view was analysed (several are visible in most
  fields); the cell nearest the image centre is used (an assumption, see `strobe.py`).
- In most files the first image is about 20% darker over the whole frame than the next 39. It is
  kept in F0, as section 2.3 defines F0 as the mean of the first 40 images, and a variant without
  it is shown below.
- 11 cells are in the Fig. 2C/D session where the paper reports n = 8, and 8 cells in the Fig. 4D
  session where it reports n = 10.

### Field direction and the timing of the pulses

The angle of the cathodic pole is found per session from the polarization profile at
non-electroporating fields (150 and 230 V/cm for 1 us; up to 1000 V/cm for 0.3 us, the fields Fig.
4D fits with a cosine). Then the per-image contrast between the poles should be zero for images 1 to
40 and jump at image 41, the first with a PEF (section 2.3), in PEF recordings and not in the sham
recordings. That ties the image index to the pulse timing.

In [ ]:
s1 = pef[pef.duration_us == 1.0]
s3 = pef[(pef.duration_us == 0.3) & pef.field.notna()]
phi1, ph1 = strobe.session_phase(s1, res, s1.field <= 230)
phi3, ph3 = strobe.session_phase(s3, res, s3.field <= 1000)
print(f"cathodic pole angle, 1 us session: {phi1:.1f} deg (cells {ph1.phase_deg.min():.1f} to {ph1.phase_deg.max():.1f}); "
      f"0.3 us session: {phi3:.1f} deg (cells {ph3.phase_deg.min():.1f} to {ph3.phase_deg.max():.1f})")
print("(90 deg is straight up in the image; Fig. 1 shows the cathodic side up)")
print(pd.concat([ph1, ph3]).round(2).to_string(index=False))

sham = files[files.kind == "sham"]
for name, sub, phi in [("1 us PEF", s1, phi1), ("0.3 us PEF", s3, phi3),
                       ("sham 2025-05-01", sham[sham.session_id == "session_2025-05-01"], phi1),
                       ("sham 2025-05-06", sham[sham.session_id == "session_2025-05-06"], phi3)]:
    t = strobe.per_image_table(sub, res, phi, "none")
    t["contrast"] = t.cathodic - t.anodic
    m = t.groupby("image").contrast.mean().to_numpy()
    jump = np.argmax(np.diff(m)) + 1
    print(f"{name:16s} n={sub.shape[0]:3d}: cathodic minus anodic dF, images 1-40 {m[:40].mean():6.2f}%, "
          f"41-100 {m[40:].mean():6.2f}%; largest step between images at index {jump} (image {jump + 1})")

### Background readings

Section 2.3 describes no background subtraction for the potentiometric images, while section 2.4
subtracts a cell-free region for Fluo-4; the claims file names both readings. Three are run:
`none`; `annulus`, the per-image median of a cell-free ring 2 to 3 cell radii from the centre with
other cells masked out; and `bckg_file`, the mean of the cell-free background recording made for
each 2025-05-01 cell, point by point along the trace. Below, the levels each reading removes.

In [ ]:
bk_assets = files[files.kind == "background"].set_index("cell").asset_id.to_dict()
bk = {cell: strobe.background_trace(res, a) for cell, a in bk_assets.items()}
lv = []
for r in cells.itertuples():
    R = res[r.asset_id]
    lv.append(dict(session=r.session_id, trace=R["trace"][:40].mean(), annulus=np.nanmean(R["bg"][:40]),
                   bckg_file=np.nan if r.cell not in bk else bk[r.cell].mean()))
lv = pd.DataFrame(lv)
print("mean baseline intensity (camera counts): trace, annulus background, background recording")
print(lv.groupby("session").agg(["mean", "min", "max"]).round(1).to_string())
print("background as a fraction of the trace intensity:",
      lv.groupby("session").apply(lambda g: (g.annulus / g.trace).mean()).round(3).to_dict())

## Sets of 150 and 230 V/cm before and after 1000 V/cm (Fig. 2C and D)

Per image, pole dF against F0, the mean of the set's first 40 images, in the +-5 degree arcs
(section 2.3); set means over the 60 PEF images. Sets are numbered within each field, before and
after the 1000 V/cm set; the k-th set at a field is compared, across the cells that have one, with
the first set at the same field (the Fig. 2 legend: "Dashed lines mark the means for the first
trial ... 2-tailed paired t-test").

In [ ]:
BG_MODES = ["annulus", "none", "bckg_file"]


def set_table(bg_mode, f0_from=0):
    t = strobe.per_image_table(s1, res, phi1, bg_mode, bk=bk, f0_from=f0_from)
    t1000 = t[t.field == 1000].groupby("cell").order.first()
    t["post"] = t.order > t.cell.map(t1000)
    pefimg = t[t.image >= 40]
    s = pefimg.groupby(["cell", "asset_id", "field", "order", "post"])[["cathodic", "anodic"]].mean().reset_index()
    s = s.sort_values(["cell", "order"])
    s["k"] = s.groupby(["cell", "field", "post"]).cumcount()
    s["both"] = (s.cathodic - s.anodic) / 2   # mean of the two poles' |dF|
    return t, s


tabs = {m: set_table(m) for m in BG_MODES}
tabs["annulus_f0_2to40"] = set_table("annulus", f0_from=1)
tabs["none_f0_2to40"] = set_table("none", f0_from=1)

for m, (t, s) in tabs.items():
    g = s.groupby(["field", "post", "k"]).agg(n=("cathodic", "size"), cathodic=("cathodic", "mean"),
                                              cathodic_se=("cathodic", "sem"), anodic=("anodic", "mean"),
                                              anodic_se=("anodic", "sem"))
    print(f"--- background reading: {m}; set means of pole dF (%), mean and s.e. over cells")
    print(g.round(2).to_string())

In [ ]:
def compare_to_first(s, pole):
    # each later set against the first set of the same field, paired over cells
    first = s[(~s.post) & (s.k == 0)].set_index(["cell", "field"])[pole]
    rows = []
    for (field, post, k), g in s.groupby(["field", "post", "k"]):
        if field == 1000 or (not post and k == 0):
            continue
        g = g.set_index(["cell", "field"])[pole]
        idx = g.index.intersection(first.index)
        x, y = g.loc[idx].to_numpy(), first.loc[idx].to_numpy()
        p = stats.ttest_rel(x, y).pvalue if len(idx) >= 3 else np.nan
        rows.append(dict(field=int(field), when="after" if post else "before", k=k, n=len(idx), pole=pole,
                         first=y.mean(), this=x.mean(), diff=(x - y).mean(),
                         reduction_pct=100 * (1 - x.mean() / y.mean()),
                         reduction_pct_per_cell=np.mean(100 * (1 - x / y)), p=p))
    return pd.DataFrame(rows)


cmp = {m: pd.concat([compare_to_first(s, pole) for pole in ["cathodic", "anodic", "both"]], ignore_index=True)
       for m, (t, s) in tabs.items()}

### fig2cd-nonelectroporating-stable

> Repeated alternating sets of 150 and 230 V/cm PEFs produced dF responses that stayed constant
> within each set and between sets, so that 800 flashes and 480 pulses had no obvious impact before
> the 1000 V/cm exposure.

Criterion: a claim of absence; each 150 or 230 V/cm set before 1000 V/cm against the first set of
its field, 2-tailed paired t-test at p < 0.05, at either pole. Reproduced when none differs. The
within-set change (first against last 10 PEF images) and the differences in percent are variants.
Sets compared with fewer than 3 cells are not tested.

In [ ]:
for m in BG_MODES + ["annulus_f0_2to40", "none_f0_2to40"]:
    c = cmp[m]
    pre = c[(c.when == "before") & c.pole.isin(["cathodic", "anodic"]) & (c.n >= 3)]
    sig = pre[pre.p < 0.05]
    print(f"--- {m}: {len(pre)} pre-exposure comparisons, {len(sig)} with p < 0.05; smallest p {pre.p.min():.3f}")
    print(pre[["field", "k", "n", "pole", "first", "this", "diff", "reduction_pct", "p"]].round(3).to_string(index=False))

print("\nwithin-set change, PEF images 51-60 minus 1-10, before 1000 V/cm (annulus reading):")
t, s = tabs["annulus"]
pre_sets = t[(~t.post) & (t.field < 1000)]
w = pre_sets.assign(part=np.where(pre_sets.image.between(40, 49), "first10",
                                  np.where(pre_sets.image >= 90, "last10", "other")))
w = w[w.part != "other"].groupby(["asset_id", "cell", "field", "part"])[["cathodic", "anodic"]].mean().unstack("part")
for pole in ["cathodic", "anodic"]:
    d = w[(pole, "last10")] - w[(pole, "first10")]
    for f in [150, 230]:
        dd = d[d.index.get_level_values("field") == f]
        per_cell = dd.groupby(level="cell").mean()
        print(f"  {pole:9s} {f} V/cm: mean change {dd.mean():.2f}% over {len(dd)} sets "
              f"(per-cell means, one-sample t-test p = {stats.ttest_1samp(per_cell, 0).pvalue:.3f}, n = {len(per_cell)})")

In [ ]:
t, s = tabs["annulus"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for i, pole in enumerate(["cathodic", "anodic"]):
    for f, col in [(150, "tab:blue"), (230, "tab:green"), (1000, "tab:red")]:
        g = s[s.field == f].groupby("order")[pole].agg(["mean", "sem"])
        ax[i].errorbar(g.index, g["mean"], g["sem"], fmt="o", color=col, label=f"{f} V/cm")
    ax[i].axhline(0, color="k", lw=0.5)
    ax[i].set_xlabel("recording number in the cell's sequence (0 = sham)")
    ax[i].set_ylabel(f"{pole} pole dF, %")
    ax[i].legend(fontsize=8)
plt.suptitle("set means, annulus background reading (mean and s.e. over cells)")
plt.tight_layout()
plt.show()
gm = s.groupby(["field", "order"])[["cathodic", "anodic"]].mean()
print("plotted group means range", round(gm.cathodic.min(), 1), "to", round(gm.cathodic.max(), 1), "(cathodic) and",
      round(gm.anodic.min(), 1), "to", round(gm.anodic.max(), 1), "(anodic); all within the axes")
print("\nper cell, cathodic set means (annulus reading), first 150 V/cm set and mean of the later pre-exposure 150 V/cm sets:")
pc = s[(s.field == 150) & ~s.post]
print(pd.DataFrame({"first": pc[pc.k == 0].set_index("cell").cathodic,
                    "later_mean": pc[pc.k > 0].groupby("cell").cathodic.mean()}).round(1).to_string())

**Result: not reproduced, resting on the background assumption.** With the cell-free annulus subtracted (the adopted reading), 1 of 12 pre-exposure comparisons differs from the first set of its field at p < 0.05: the third 150 V/cm set at the cathodic pole, 10.2% against 15.4% (p = 0.047, n = 11). With the background recording subtracted, 2 of 12 have p < 0.05 (smallest p 0.034). Without background subtraction none does (smallest p 0.057), so the outcome depends on the reading. With F0 from images 2 to 40, 2 of 12 differ under both readings. Under the annulus reading every later pre-exposure 150 V/cm set is 26 to 34% below the first at the cathodic pole, while the 230 V/cm sets differ from their first by -8.5 to 8.1% at that pole. Within sets, the mean change from the first to the last 10 PEF images is under 1% at either pole and field (p 0.37 or more).

### fig2c-1000-within-set

> At 1000 V/cm, cathodic pole dF fell from 40 +- 10% at the start of the set to 26.6 +- 8% at its
> end, a trend that was not statistically significant.

Criterion: readings of the beginning and end (a) first and last PEF image, (b) PEF images 1 to 10
and 51 to 60, (c) 1 to 5 and 56 to 60. Reproduced when, under a reading, both means are within
tolerance (30 to 50% and 18.6 to 34.6%), the end is lower, and a 2-tailed paired t-test across cells
gives p > 0.05.

In [ ]:
WINDOWS = {"a: first and last image": ([40], [99]), "b: images 1-10 and 51-60": (range(40, 50), range(90, 100)),
           "c: images 1-5 and 56-60": (range(40, 45), range(95, 100))}
rows = []
for m in BG_MODES + ["annulus_f0_2to40", "none_f0_2to40"]:
    t, s = tabs[m]
    x = t[t.field == 1000]
    for wname, (wa, wb) in WINDOWS.items():
        a = x[x.image.isin(list(wa))].groupby("cell").cathodic.mean()
        b = x[x.image.isin(list(wb))].groupby("cell").cathodic.mean()
        p = stats.ttest_rel(a, b).pvalue
        ok = (abs(a.mean() - 40) <= 10) and (abs(b.mean() - 26.6) <= 8) and (b.mean() < a.mean()) and (p > 0.05)
        rows.append(dict(background=m, reading=wname, n=len(a), start=a.mean(), start_se=a.sem(), end=b.mean(),
                         end_se=b.sem(), p=p, start_in_tol=abs(a.mean() - 40) <= 10, end_in_tol=abs(b.mean() - 26.6) <= 8,
                         criterion_met=ok))
r1000 = pd.DataFrame(rows)
print(r1000.round(3).to_string(index=False))

t, s = tabs["annulus"]
x = t[t.field == 1000]
print("\nper cell, annulus reading, mean of PEF images 1-10 and 51-60 (cathodic, %):")
pc = pd.DataFrame({"start": x[x.image.between(40, 49)].groupby("cell").cathodic.mean(),
                   "end": x[x.image.between(90, 99)].groupby("cell").cathodic.mean()})
print(pc.round(1).to_string())
print(f"cells where the end is lower: {(pc.end < pc.start).sum()} of {len(pc)}")

**Result: reproduced, resting on the background assumption.** Under the annulus reading and window (b), cathodic dF falls from 31.9 +- 5.1% to 21.0 +- 5.1% (n = 11, p = 0.051), both within tolerance, and the end is lower in 8 of 11 cells. Windows (a) and (c) also put both values within tolerance (39.6 to 21.0% and 34.9 to 23.0%) but give p = 0.009 and 0.002. The background recording gives the same result under (b) (30.1 to 20.5%, p = 0.126). Without background subtraction the values are 12.6 and 8.2% under (b), and no window matches. With F0 from images 2 to 40 the start under (b) is 29.6%, just outside the tolerance. The matching reading's p-value is at the edge of 0.05.

### fig2d-lasting-reduction

> Set-averaged dF stayed nearly constant before electroporation but dropped by about 20% (a small but
> persistent reduction of at most 20%) in the 150 and 230 V/cm sets acquired after the 1000 V/cm
> exposure.

Criterion: each set against the first set of its field, 2-tailed paired t-test across cells;
readings (a) each pole, (b) both poles' |dF| averaged; size (i) the mean reduction over all
post-1000 V/cm sets, (ii) each set's reduction. Reproduced when, under a reading, the post-exposure
sets show a reduction within 18 to 22% with p < 0.05 for at least one set, and no pre-exposure set
differs at p < 0.05. Partially reproduced when the reduction is significant but its size is outside
the tolerance under every reading. Reduction is 100 (1 - set mean / first-set mean) of the group
means; the mean of per-cell reductions is shown too. Sets after 1000 V/cm compared with fewer
than 3 cells (the third 150 V/cm set, in 2 cells) are not tested.

In [ ]:
summary = []
for m in BG_MODES + ["annulus_f0_2to40", "none_f0_2to40"]:
    c = cmp[m]
    for pole in ["cathodic", "anodic", "both"]:
        post = c[(c.when == "after") & (c.pole == pole) & (c.n >= 3)]
        pre = c[(c.when == "before") & (c.pole == pole) & (c.n >= 3)]
        if m == BG_MODES[0] or pole == "both":
            pass
        summary.append(dict(background=m, pole=pole, post_sets=len(post),
                            mean_reduction=post.reduction_pct.mean(),
                            set_reductions=", ".join(f"{v:.1f}" for v in post.reduction_pct),
                            set_p=", ".join(f"{v:.3f}" for v in post.p),
                            any_post_p05=(post.p < 0.05).any(),
                            any_in_tol_each=(post.reduction_pct.between(18, 22)).any(),
                            in_tol_and_p05_same_set=((post.reduction_pct.between(18, 22)) & (post.p < 0.05)).any(),
                            mean_in_tol=18 <= post.reduction_pct.mean() <= 22,
                            pre_min_p=pre.p.min(), pre_any_p05=(pre.p < 0.05).any()))
summ = pd.DataFrame(summary)
print(summ.round(3).to_string(index=False))
print("\npost-exposure comparisons, annulus reading:")
c = cmp["annulus"]
print(c[c.when == "after"][["field", "k", "n", "pole", "first", "this", "reduction_pct", "reduction_pct_per_cell", "p"]]
      .round(3).to_string(index=False))

**Result: partially reproduced.** After 1000 V/cm, the mean of the two poles' |dF| is 33.2%, 29.2% and 25.7% below the first set of the same field in the first three 150 and 230 V/cm sets (p = 0.010, 0.021 and 0.024, annulus reading) and 21.8% below in the fourth (p = 0.149); the mean reduction over the four sets is 27.5%, outside 18 to 22%. Without background subtraction the four reductions are 23.8 to 31.3% (mean 28.1%). With the paper's F0, no combination of background, pole and size reading has a set whose reduction is within 18 to 22% with p < 0.05. At single poles, the cathodic pole carries the drop at 150 V/cm (53.4% and 40.8%) and the anodic pole at 230 V/cm (29.9% and 28.4%). The condition that no pre-exposure set differs fails under the annulus reading (p = 0.047) and holds without subtraction. The reduction is present and significant under every reading but larger than about 20%.

## 0.3 us pulses (Fig. 4D)

### fig4d-short-pulses-same-limit

> With 0.3 us PEFs at 357 to 3320 V/cm (imaged at 275 ns), the maximum dF was 53 +- 4% (about 600
> mV) at both poles, similar to 1 us pulses, and its pattern with increasing field strength did not
> differ from that with 1 us pulses.

Criterion: pole dF from the polarization image (mean of the 60 PEF images against the mean of the
40 baseline images), mean over cells. Reproduced when the maximum |dF| over fields is within 49 to
57% at both poles, and pole |dF| declines at fields beyond the maximum with the larger fractional
decline at the cathodic pole. Partially reproduced when one pole's maximum is out of tolerance.
The fractional decline is taken from each pole's maximum to its value at the strongest field.

In [ ]:
cath3, anod3 = strobe.pole_index(phi3)
rows4 = []
for m in ["annulus", "none"]:
    prof = strobe.profile_table(s3, res, m)
    pt = pd.DataFrame([dict(cell=r.cell, field=r.field, cathodic=prof[r.asset_id][cath3].mean(),
                            anodic=prof[r.asset_id][anod3].mean()) for r in s3.itertuples()])
    g = pt.groupby("field").agg(n=("cathodic", "size"), cathodic=("cathodic", "mean"), cathodic_se=("cathodic", "sem"),
                                anodic=("anodic", "mean"), anodic_se=("anodic", "sem"))
    print(f"--- background reading {m}: pole dF (%) per field, mean and s.e. over cells")
    print(g.round(2).to_string())
    out = dict(background=m)
    for pole in ["cathodic", "anodic"]:
        v = g[pole].abs()
        fmax = v.idxmax()
        beyond = v[v.index > fmax]
        out[f"{pole}_max"] = v.max()
        out[f"{pole}_max_field"] = fmax
        out[f"{pole}_max_in_tol"] = abs(v.max() - 53) <= 4
        out[f"{pole}_declines_beyond"] = bool((np.diff(np.r_[v.max(), beyond.to_numpy()]) < 0).all())
        out[f"{pole}_frac_decline_to_3320"] = 1 - v.iloc[-1] / v.max()
    rows4.append(out)
    print("per cell, maximum |dF| over fields:")
    print(pt.assign(cathodic=pt.cathodic.abs(), anodic=pt.anodic.abs()).groupby("cell")[["cathodic", "anodic"]].max().round(1).to_string())
r4 = pd.DataFrame(rows4)
print()
print(r4.round(3).T.to_string())

In [ ]:
prof = strobe.profile_table(s3, res, "annulus")
ang = strobe.relative_angle(phi3)
order_idx = np.argsort(ang)
fields = sorted(s3.field.unique())
fig, ax = plt.subplots(1, len(fields), figsize=(2.0 * len(fields), 2.6), sharey=True)
lo, hi = 0, 0
for i, f in enumerate(fields):
    P = np.array([prof[a] for a in s3[s3.field == f].asset_id])
    m = P.mean(0)
    lo, hi = min(lo, m.min()), max(hi, m.max())
    ax[i].plot(ang[order_idx], m[order_idx], lw=0.8)
    ax[i].axhline(0, color="k", lw=0.4)
    ax[i].set_title(f"{int(f)} V/cm", fontsize=8)
    ax[i].set_xticks([-180, 0, 180])
ax[0].set_ylabel("dF, % (annulus background)")
fig.supxlabel("polar angle from the cathodic pole, degrees", fontsize=9)
plt.tight_layout()
plt.show()
print(f"range of the plotted mean profiles: {lo:.1f} to {hi:.1f}%")

**Result: not reproduced.** The maximum mean |dF| over fields is 34.9% at the cathodic pole (1200 V/cm) and 34.0% at the anodic pole (1630 V/cm) with the annulus background subtracted, and 15.0% and 16.7% without it; all are outside 49 to 57%. Three of the 8 cells reach 49.6 to 52.4% at the cathodic pole (annulus reading). The pattern holds. Beyond the maximum, |dF| falls at every step at both poles, and by 3320 V/cm it is 82% below the maximum at the cathodic pole and 49% below at the anodic pole, so the decline is larger at the cathodic pole. The dataset holds 8 cells where the paper reports 10.

## Eq. (2) (Fig. 5B)

### fig5b-eq2-sensitivity

> By Eq. (2), effective membrane conductivity has little effect on charging while it stays much
> smaller than the cytosol and medium conductivities; a 10-fold increase gives about a 2% IMP
> reduction and a 100-fold increase about 12%.

Criterion: Eq. (2) with cytoplasm 0.4 S/m, medium 1.64 S/m, intact membrane 3e-7 S/m, R = 8 um and
d = 5 nm; reduction 1 - f/1.5 at 10 and 100 times the intact conductivity; both within tolerance
(1.8 to 2.2% and 10.8 to 13.2%). Eq. (2) is taken as printed in the PDF (section 2.3), which the
extracted text garbles. If it gave f far from 1.5 at the intact conductivity, a normalized form
would also be shown; f is 1.498 there, so the literal form and the normalized one (reduction
against f at the intact value) are both shown.

In [ ]:
f_intact = float(strobe.f_eq2(strobe.LAMBDA_M0))
print(f"f at the intact membrane conductivity: {f_intact:.5f}")
rows5 = []
for k in [1, 2, 3, 5, 10, 20, 30, 50, 100, 200, 1000]:
    f = float(strobe.f_eq2(strobe.LAMBDA_M0 * k))
    rows5.append(dict(fold=k, lambda_m_mS_per_m=1e3 * strobe.LAMBDA_M0 * k, f=f,
                      reduction_vs_1p5_pct=100 * (1 - f / 1.5), reduction_vs_intact_pct=100 * (1 - f / f_intact)))
r5 = pd.DataFrame(rows5)
print(r5.round(4).to_string(index=False))
for k, lo_, hi_ in [(10, 1.8, 2.2), (100, 10.8, 13.2)]:
    v = r5.set_index("fold").loc[k]
    print(f"{k}-fold: {v.reduction_vs_1p5_pct:.2f}% (literal), {v.reduction_vs_intact_pct:.2f}% (normalized); "
          f"tolerance {lo_} to {hi_}% -> {lo_ <= v.reduction_vs_1p5_pct <= hi_} / {lo_ <= v.reduction_vs_intact_pct <= hi_}")
k2 = np.interp(2.0, r5.reduction_vs_1p5_pct, r5.fold)
print(f"fold increase giving a 2% reduction (interpolated on the table): {k2:.1f}")

**Result: partially reproduced.** Eq. (2) as printed gives f = 1.498 at the intact conductivity. A 100-fold increase reduces IMP by 11.87% (11.75% against f at the intact value), within 10.8 to 13.2%. A 10-fold increase reduces it by 1.33% (1.20%), outside 1.8 to 2.2%; a 2% reduction needs about a 15-fold increase. Both are small next to the 57.4% reduction at 1000-fold, as the statement says.

## Summary

| Claim | Outcome |
|---|---|
| fig2cd-nonelectroporating-stable | not reproduced (rests on the background assumption; reproduced without subtraction) |
| fig2c-1000-within-set | reproduced (rests on the background assumption) |
| fig2d-lasting-reduction | partially reproduced |
| fig4d-short-pulses-same-limit | not reproduced |
| fig5b-eq2-sensitivity | partially reproduced |
| the other 25 claims | blocked, data missing (FluoVolt, 1 us escalation, Gd3+, Ca2+ and two-pulse experiments are not in the dataset) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The membrane trace is found automatically (the outer ridge of the staining in the sham
  recording's baseline image, moved by the measured drift of at most 2 pixels), where the paper
  draws it by hand; its 360 points are spaced by angle, not by arc length.
- The analysed cell is the one nearest the image centre; the files do not say which cell was used.
- The cathodic pole is the session's median cosine phase (83.0 and 82.1 degrees, close to straight
  up, as in Fig. 1).
- A cell-free background is subtracted (the adopted reading; section 2.3 describes none). The
  paper's values at 1000 V/cm (40 and 26.6% in section 3.2) are on the scale of the subtracted
  readings; without subtraction dF is about 40% of that, since background is
  52% of the trace intensity.
- All 11 cells of the 2025-05-01 session are used (the paper reports n = 8), and all 8 of
  2025-05-06 (the paper reports n = 10).
- Field strengths come from the paper's labels matched to the files' amplitudes by order.

What was found about the dataset:

- No field strengths, only pulse amplitudes in volts; electrode distance 1.00 mm against the
  paper's 1.6 mm.
- One general description, of the 2025-05-01 protocol, is in every file, including the 0.3 us
  escalation files of 2025-05-06; it also calls each set "a single pulse".
- `starting_time` is a Unix time; no pixel size; no record of which cell was analysed.
- The first image of most files is about 22% darker over the whole frame (174 of 232 cell
  files).
- 11 and 8 cells against the paper's 8 and 10; the dandiset's title and description mention only
  the post-electroporation experiment.
- The image timing checks out: the pole contrast is 0 in images 1 to 40 and steps at image 41
  in PEF recordings, not in shams.